In [1]:
!wget -O - https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt > ../../data/raw/sheakspear_input.txt

--2025-06-04 14:40:30--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: 'STDOUT'

-                   100%[===================>]   1.06M  --.-KB/s    in 0.01s   

2025-06-04 14:40:30 (87.6 MB/s) - written to stdout [1115394/1115394]



In [2]:
with open('../../data/raw/sheakspear_input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [3]:
len(text)

1115394

In [4]:
chars = sorted(list(set(text)))
vocab_size = len(chars)

In [5]:
chars

['\n',
 ' ',
 '!',
 '$',
 '&',
 "'",
 ',',
 '-',
 '.',
 '3',
 ':',
 ';',
 '?',
 'A',
 'B',
 'C',
 'D',
 'E',
 'F',
 'G',
 'H',
 'I',
 'J',
 'K',
 'L',
 'M',
 'N',
 'O',
 'P',
 'Q',
 'R',
 'S',
 'T',
 'U',
 'V',
 'W',
 'X',
 'Y',
 'Z',
 'a',
 'b',
 'c',
 'd',
 'e',
 'f',
 'g',
 'h',
 'i',
 'j',
 'k',
 'l',
 'm',
 'n',
 'o',
 'p',
 'q',
 'r',
 's',
 't',
 'u',
 'v',
 'w',
 'x',
 'y',
 'z']

In [6]:
print(''.join(chars))


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz


In [7]:
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda s: ''.join(itos[c] for c in s)

In [8]:
print(encode("hello world"))
print(decode([46, 43, 50, 50, 53, 1, 61, 53, 56, 50, 42]))

[46, 43, 50, 50, 53, 1, 61, 53, 56, 50, 42]
hello world


In [9]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
x = torch.randn(3, 3).to(device)

Device: cuda


In [10]:
data = torch.tensor(encode(text), dtype=torch.long, device=device)

In [11]:
data

tensor([18, 47, 56,  ..., 45,  8,  0], device='cuda:0')

In [12]:
n = int(0.9 * len(data))
train_data = data[:n]
test_data = data[n:]

In [13]:
block_size = 8
train_data[:block_size + 1]


tensor([18, 47, 56, 57, 58,  1, 15, 47, 58], device='cuda:0')

In [14]:
x = train_data[:block_size]
y = train_data[1:block_size + 1]

In [15]:
print(x)
print(y)

tensor([18, 47, 56, 57, 58,  1, 15, 47], device='cuda:0')
tensor([47, 56, 57, 58,  1, 15, 47, 58], device='cuda:0')


In [16]:

torch.manual_seed(1337)
batch_size = 4
block_size = 8

def get_batch(split):
    data = train_data if split == 'train' else test_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = []
    y = []
    for i in ix:
        x.append(data[i: i + block_size])
        y.append(data[i + 1: i + block_size + 1])
    x = torch.stack(x)
    y = torch.stack(y)
    return x, y
    
xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('----')

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]], device='cuda:0')
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]], device='cuda:0')
----


In [17]:
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size_):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size_, vocab_size_)
        
    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)
        
        if targets is None:
            loss = None
        else :
            B, T, C = logits.shape
            
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss
    
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self(idx)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            
            id_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, id_next), dim=1)
        return idx
        
m = BigramLanguageModel(vocab_size_=vocab_size).to(device)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)
r = decode(m.generate(torch.zeros((1, 1), dtype=torch.long, device=device), max_new_tokens=50)[0].tolist())
print(r)

torch.Size([32, 65])
tensor(4.8786, device='cuda:0', grad_fn=<NllLossBackward0>)

pYCXxfRkRZd
wc'wfNfT;OLlTEeC K
jxqPToTb?bXAUG:C-SG


In [18]:
optimizer = torch.optim.Adam(m.parameters(), lr=1e-3)

In [19]:
for steps in range(100000):
    xb, yb = get_batch('train')
    
    optimizer.zero_grad(set_to_none=True)
    logits, loss = m(xb, yb)
    loss.backward()
    optimizer.step()
print(loss.item())

2.6846251487731934


In [20]:
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long, device=device), max_new_tokens=500)[0].tolist()))


BO:
IS:
Falatanss:
Wanthar u qur, vet?
F dilasoate awice my.
Whandacom oroup
Yowhthetof isth ble mil ndill, ath iree senghin lat Heridrovets, and Win nghire yombousel lind me l.
HAshe ce hiry:
Supr aisspll, y.
Herindu n Boopetelaves
MPOLI s, d mothakleo Windo whth eisbyo the m dourive we higend t so mower; te

AN ad nterupirf s ar igr t m:

Thiny aleronth,
Mad re?

WISo myoffursuk!
KENoby ak
Sadsal thes ghesthidin cour ay aney IOUSts I fr y ce.
Jongheand, bemary.
Yof 'sour mend sora anghy t--pon


In [23]:
torch.manual_seed(1337)

B, T, C = 4, 8, 32

x = torch.randn(B, T, C).to(device)

head_size = 16

key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

k = key(x)
q = query(x)
v = value(x)

qk = q @ k.transpose(-2, -1)

tril = torch.tril(torch.ones(T, T))

qk = qk.masked_fill(tril == 0, float('-inf'))
qk = F.softmax(qk, dim=-1)

attention = qk @ v
print(attention.shape)

RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu! (when checking argument for argument mat2 in method wrapper_CUDA_mm)

NameError: name 'attention' is not defined